# Nsquare | 반도체 불량 분류 실습 — **XGBoost와 MLP 비교하기**

**목표:** 동일한 SECOM 데이터에서 **XGBoost와 PyTorch MLP를 모두 학습·비교**하고, 자유롭게 개선한 뒤 **Validation Macro-F1이 가장 높은 모델 하나**를 선택합니다. 최종 모델은 XGBoost 또는 MLP 중 어느 쪽을 사용해도 됩니다.

- **실습 A:** XGBoost — 결정 트리 기반 반도체 불량 분류
- **실습 B:** PyTorch MLP — 결측치 처리·표준화 후 신경망 학습
- **실습 C:** 자유 실험 — 전처리, 구조, Loss, 학습 설정, 임계값 등을 바꾸며 성능 개선
- **실습 D:** 설정 확정 → Test 최종 1회 평가 → 정량·정성 결과보고서 작성
- **공통 평가:** Macro-F1, Accuracy, Balanced Accuracy, 불량 Precision / Recall / F1, Average Precision(AP)

**데이터 출처:** [UCI SECOM](https://archive.ics.uci.edu/dataset/179/secom). `AILAB_w8.ipynb`의 과제 목표와 고정 평가 규칙을 유지한 Nsquare 실습 버전입니다.


## 1. 공통 조건과 실행 순서

| 항목 | 이 실습의 공통 조건 |
|---|---|
| 실행 환경 | RTX 5080이 있는 로컬 PC의 Jupyter / VS Code에서 실행 가능. **기본 학습은 CPU**, MLP의 CUDA 사용은 선택 |
| 데이터 | SECOM **1,567개 샘플 · 590개 센서**. 라벨 파일의 시간 정보는 입력에서 제외 |
| 라벨 | 원본 `−1 → 0` 정상(Pass), 원본 `+1 → 1` 불량(Fail). 불량을 양성 클래스로 정의 |
| 고정 분할 | Train / Validation / Test ≈ **70 / 10 / 20**, 분할 시드 **42**. 원본과 동일한 두 단계 계층 분할 |
| 선택 기준 | **Validation Macro-F1 최대화**. Test 점수로 모델·설정·시드·임계값을 선택하지 않음 |
| 학습 범위 | 전처리와 모델은 **Train만** 사용. **Train + Validation 재학습 금지** |
| 비교·최종 선택 | XGBoost와 MLP를 모두 비교한 뒤 **둘 중 하나**를 선택. 두 계열을 결합한 앙상블은 사용하지 않음 |
| 변경 가능 범위 | Loss, 구조, 목적함수, 전처리, 학습 샘플링, optimizer, scheduler, 정규화, 학습 시드, epoch, 임계값 등 |
| 실험 자유도 | **필수 실험 종류나 최소 횟수는 없음**. 제공 코드만 실행하는 데서 끝내지 말고 직접 실험·해석 |
| 결과 저장 | `secom_lab_runs/`에 설정, 분할 인덱스, 비교 CSV, 그래프, 최종 모델 저장 |

**실행 순서**

1. **2~4절:** 환경 준비 → SECOM 로딩 → 고정 분할 및 Train 데이터 확인.
2. **5~8절:** 공통 평가·전처리 준비 → XGBoost와 MLP baseline 학습 → Validation 비교.
3. **9~10절:** 자유 실험 및 결과 분석. 임계값 탐색은 원할 때만 `TUNE_THRESHOLD=True`로 설정합니다.
4. **11~12절:** 모든 선택을 마친 뒤 `FINAL_SETTINGS_CONFIRMED=True`, `RUN_FINAL_TEST=True`로 바꾸고 두 절의 코드 셀을 차례로 실행합니다.
5. **13절:** 결과보고서를 작성하고 실행 결과가 포함된 노트북을 저장합니다.

**참가자 / 팀**

| 팀 이름 | 이름 | 역할 |
|---|---|---|
| 작성 | 작성 | 작성 |

제출 방법과 일정은 Nsquare 운영 안내를 따릅니다. 제출 파일명 예: `Nsquare_SECOM_팀명.ipynb`.


## 2. 실행 환경과 공통 설정

**질문:** GPU가 있는 PC에서도 작은 표 형식 데이터 실습을 CPU로 진행할 수 있을까요?

이 노트북은 원본과 동일하게 **CPU·2개 스레드**를 기본값으로 사용합니다. RTX 5080은 필수 조건이 아닙니다. MLP를 GPU에서 실행하려면 아래 `CONFIG["mlp_device"]`를 `"cuda"`로 바꾸세요. GPU 사용 여부는 평가 점수에 영향을 주는 가산 조건이 아니며, CPU/GPU와 라이브러리 버전이 달라지면 결과도 달라질 수 있습니다.

설치 셀은 **없는 패키지만 설치**하며, 이미 설치된 PyTorch를 일괄 교체하지 않습니다. PyTorch가 없다면 기본 배포판을 설치합니다. CUDA 실행에 필요한 설치 명령은 [PyTorch 공식 설치 안내](https://pytorch.org/get-started/locally/)에서 PC 환경에 맞게 확인하세요. 설치 후 커널 재시작 안내가 나오면 재시작한 뒤 처음부터 실행합니다.


In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    "numpy": "numpy>=1.24,<3",
    "pandas": "pandas>=2,<3",
    "sklearn": "scikit-learn>=1.3,<2",
    "xgboost": "xgboost>=2,<4",
    "matplotlib": "matplotlib>=3.7,<4",
    "joblib": "joblib>=1.3,<2",
    "torch": "torch>=2,<3",
}
missing = [spec for module, spec in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("필요한 패키지 설치:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("필요한 Python 라이브러리가 이미 설치되어 있습니다.")


In [ ]:
from pathlib import Path
import copy
import io
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost
import joblib
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Image
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, average_precision_score,
    ConfusionMatrixDisplay, classification_report)
from xgboost import XGBClassifier


import hashlib
import json
import platform
import random
import sys
import time
from datetime import datetime, timezone

SPLIT_SEED = 42  # 고정: 원본과 같은 Train / Validation / Test 행을 사용합니다.
CONFIG = {
    "train_seed": 42,          # 학습 시드는 자유 실험에서 변경 가능
    "cpu_threads": 2,
    "mlp_device": "cpu",       # "cpu" 또는 "cuda"; 기본값은 원본과 동일
    "data_dir": "dataset/secom",
    "run_dir": "secom_lab_runs",
}
TRAIN_SEED = CONFIG["train_seed"]
DATA_DIR = Path(CONFIG["data_dir"])
DATA_ROOT = DATA_DIR
RUN_DIR = Path(CONFIG["run_dir"])
for folder in (DATA_DIR, RUN_DIR):
    folder.mkdir(parents=True, exist_ok=True)

def save_output(filename, writer):
    # 최종 평가가 끝난 폴더의 설정·표·그림을 이후 실행으로 덮어쓰지 않습니다.
    if (RUN_DIR / "SECOM_final_completed.json").exists():
        return False
    writer(RUN_DIR / filename)
    return True

if (RUN_DIR / "SECOM_final_completed.json").exists():
    print("이 폴더에는 완료된 최종 평가가 있습니다. 기존 파일을 보존합니다.")
    print("재실행한 Validation 출력은 저장하지 않으며, Test 단계에서는 기존 결과를 표시합니다.")
    print("Test 결과를 본 이후에는 모델·설정·임계값을 다시 선택하지 마세요.")
random.seed(TRAIN_SEED)
np.random.seed(TRAIN_SEED)

VERSIONS = {
    "python": sys.version.split()[0], "numpy": np.__version__,
    "pandas": pd.__version__, "sklearn": sklearn.__version__,
    "xgboost": xgboost.__version__, "torch": torch.__version__,
}
display(pd.DataFrame([CONFIG]).T.rename(columns={0: "value"}))
display(pd.DataFrame([VERSIONS]).T.rename(columns={0: "version"}))


In [ ]:
# 실제 실행 장치 진단. CUDA를 선택한 경우 작은 연산까지 확인합니다.
if CONFIG["mlp_device"] not in {"cpu", "cuda"}:
    raise ValueError('mlp_device는 "cpu" 또는 "cuda"로 설정하세요.')
if CONFIG["cpu_threads"] < 1:
    raise ValueError("cpu_threads는 1 이상이어야 합니다.")
MLP_DEVICE = torch.device(CONFIG["mlp_device"])
cuda_available = torch.cuda.is_available()
print("Python:", sys.version.split()[0], "| OS:", platform.platform())
print("PyTorch CUDA build:", torch.version.cuda, "| CUDA available:", cuda_available)
print("XGBoost baseline: CPU | MLP:", MLP_DEVICE)
if cuda_available:
    print("GPU:", torch.cuda.get_device_name(0))
if MLP_DEVICE.type == "cuda":
    if not cuda_available:
        raise RuntimeError('CUDA를 사용할 수 없습니다. PyTorch 설치를 확인하거나 mlp_device="cpu"로 바꾸세요.')
    try:
        probe = torch.ones((8, 8), device=MLP_DEVICE)
        _ = (probe @ probe).sum().item()
        torch.cuda.synchronize()
        del probe
    except Exception as exc:
        raise RuntimeError('CUDA 연산에 실패했습니다. GPU/드라이버/PyTorch 조합을 확인하거나 CPU를 사용하세요.') from exc

ENVIRONMENT = {
    "run_started_utc": datetime.now(timezone.utc).isoformat(),
    "platform": platform.platform(), "versions": VERSIONS,
    "config": copy.deepcopy(CONFIG), "split_seed": SPLIT_SEED,
    "cuda_build": torch.version.cuda, "cuda_available": cuda_available,
    "gpu": torch.cuda.get_device_name(0) if cuda_available else None,
}
save_output("SECOM_environment.json", lambda path: path.write_text(
    json.dumps(ENVIRONMENT, ensure_ascii=False, indent=2), encoding="utf-8"))
print("결과 저장 폴더:", RUN_DIR.resolve())


## 3. 원본 SECOM 데이터 준비

SECOM에는 반도체 공정의 센서 측정값, 결측값, 정상/불량 라벨이 들어 있습니다. **센서 행렬만 모델 입력으로 사용**하며, 라벨 파일에 포함된 시간 정보는 사용하지 않습니다.

- `dataset/secom/`에 `secom.data`, `secom_labels.data`가 있으면 그대로 사용합니다.
- 파일이 없으면 [UCI 원본 ZIP](https://archive.ics.uci.edu/static/public/179/secom.zip)을 내려받습니다.
- 인터넷 연결이 어려우면 두 파일을 직접 넣고 다시 실행하세요.
- 실제 센서 행렬은 **590열**입니다. 일부 데이터 소개의 특징 수 표기와 구분하여 아래 shape를 확인합니다.

**주의:** 이 실습은 무작위 계층 분할을 사용합니다. 미래 생산 시점에 대한 일반화 성능을 직접 검증하는 시간 순서 분할 실험은 아닙니다.


In [ ]:
# 원본 데이터 로더: 데이터와 라벨 정의를 유지합니다.
DATA_URL = "https://archive.ics.uci.edu/static/public/179/secom.zip"
required_files = ["secom.data", "secom_labels.data"]
if not all((DATA_ROOT / name).exists() for name in required_files):
    try:
        request = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=120) as response:
            payload = response.read()
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            for name in required_files:
                member = next(m for m in archive.namelist() if m.split("/")[-1] == name)
                (DATA_ROOT / name).write_bytes(archive.read(member))
    except Exception as exc:
        raise RuntimeError(f"Download failed. Put both SECOM files in {DATA_ROOT.resolve()}.") from exc

X = pd.read_csv(DATA_ROOT / "secom.data", sep=r"\s+", header=None)
raw_labels = pd.read_csv(DATA_ROOT / "secom_labels.data", sep=r"\s+", header=None)
assert len(X) == len(raw_labels) == 1567
assert X.shape[1] == 590, "SECOM 원본 센서 열 수를 확인하세요."
assert set(raw_labels[0].unique()) == {-1, 1}
y = raw_labels[0].map({-1: 0, 1: 1}).astype(int)
X.columns = [f"sensor_{i:03d}" for i in range(X.shape[1])]
print("Sensor matrix:", X.shape, "| Label mapping: -1 -> 0 (pass), +1 -> 1 (fail)")

DATA_HASHES = {name: hashlib.sha256((DATA_ROOT / name).read_bytes()).hexdigest()
               for name in required_files}
save_output("SECOM_data_hashes.json", lambda path: path.write_text(
    json.dumps(DATA_HASHES, indent=2), encoding="utf-8"))
print("원본 파일의 SHA-256 해시를 확인했습니다.")


## 4. 고정 분할 — 전처리보다 먼저 수행하기

**질문:** 결측치의 평균이나 표준화 기준을 전체 데이터에서 계산하면, 아직 평가하지 않은 데이터의 정보가 학습에 들어가지 않을까요?

아래 **두 단계 분할을 그대로 유지**합니다. 먼저 전체의 20%를 Test로 분리하고, 남은 80%의 1/8을 Validation으로 분리합니다. 실제 크기는 **Train 1,096 / Validation 157 / Test 314**입니다.

- 모든 실험에서 같은 행을 사용하며, 분할 시드 **42는 바꾸지 않습니다**.
- Validation/Test의 행을 삭제하거나 재표집하지 않습니다.
- 학습 데이터의 resampling은 허용되지만 **Train 내부에서만** 적용합니다. 교차검증을 추가한다면 각 학습 fold 내부에서 수행합니다.
- 데이터 분포와 결측치 분석은 Train을 중심으로 진행합니다. Test의 특징·라벨은 최종 평가 단계까지 분석에 사용하지 않습니다.


In [ ]:
# 고정 분할 — 수정하지 마세요.
all_idx = np.arange(len(y))
trainval_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, stratify=y, random_state=SPLIT_SEED)
train_idx, val_idx = train_test_split(
    trainval_idx, test_size=0.125, stratify=y.iloc[trainval_idx], random_state=SPLIT_SEED)
X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx].copy()
X_val, y_val = X.iloc[val_idx].copy(), y.iloc[val_idx].copy()
assert not (set(train_idx) & set(val_idx) or set(train_idx) & set(test_idx)
            or set(val_idx) & set(test_idx))
print("Split sizes:", {"Train": len(train_idx), "Validation": len(val_idx), "Test": len(test_idx)})
counts = pd.DataFrame({"Train": y_train.value_counts(), "Validation": y_val.value_counts()})
counts.index = ["Pass (0)" if i == 0 else "Fail (1)" for i in counts.index]
display(counts)
counts.plot.bar(rot=0, title="Class counts: Train / Validation")
plt.ylabel("Samples")
plt.tight_layout()
plt.show()
missing_pct = X_train.isna().mean().sort_values(ascending=False) * 100
display(missing_pct.head(20).rename("Train missing (%)").to_frame())
missing_pct.head(30).plot.bar(figsize=(10, 3), title="Top 30 sensors by Train missing rate")
plt.ylabel("Missing (%)")
plt.tight_layout()
plt.show()
print("Train fail ratio:", y_train.mean())

save_output("SECOM_split_indices.npz", lambda path: np.savez(
    path, train_idx=train_idx, val_idx=val_idx, test_idx=test_idx))
SPLIT_HASH = hashlib.sha256(
    json.dumps({"train": train_idx.tolist(), "val": val_idx.tolist(),
                "test": test_idx.tolist()}, sort_keys=True).encode()).hexdigest()
print("고정 분할 인덱스를 준비했습니다.")


## 5. 공통 평가 — Accuracy와 Macro-F1을 구분하기

**질문:** 대부분이 정상인 데이터에서 모두 정상으로 예측하면, Accuracy가 높더라도 불량을 찾아낸 모델이라고 할 수 있을까요?

불량(Fail = 1)을 양성 클래스로 정의합니다.

$$\mathrm{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN},\quad
P=\frac{TP}{TP+FP},\quad R=\frac{TP}{TP+FN},\quad F1=\frac{2PR}{P+R}.$$

$$\mathrm{Macro\!\! -\!\! F1}=\frac{F1_{pass}+F1_{fail}}{2},\qquad
\mathrm{Balanced\ Accuracy}=\frac{Recall_{pass}+Recall_{fail}}{2}.$$

| 지표 | 확인할 내용 |
|---|---|
| **Macro-F1** | 두 클래스의 F1을 같은 비중으로 평균. **Validation에서 최대화할 주 지표** |
| Accuracy | 전체 샘플 중 정답 비율. 정상 샘플이 많으면 높은 값이 나올 수 있음 |
| Balanced Accuracy | 정상 Recall과 불량 Recall의 평균 |
| Fail Precision | 불량 경보 중 실제 불량의 비율 |
| Fail Recall | 실제 불량 중 찾아낸 비율 |
| Fail F1 | 불량 Precision과 Recall의 조화평균 |
| AP | 연속 예측 점수의 precision–recall 순위를 요약한 보조 지표. 사다리꼴 PR-AUC와 구분 |

**고정 규칙:** `probability >= threshold`이면 불량으로 예측합니다. Macro-F1은 `labels=[0, 1], average="macro", zero_division=0`으로 계산합니다. 아래 지표 정의를 바꾸지 마세요.


In [ ]:
# 공통 평가 함수 — 정의를 유지하세요.
MAIN_METRIC = "Macro-F1"
def evaluate_scores(y_true, probabilities, threshold=0.5):
    probabilities = np.asarray(probabilities)
    assert probabilities.shape == (len(y_true),)
    assert np.isfinite(probabilities).all()
    predictions = (probabilities >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Macro-F1": f1_score(y_true, predictions, labels=[0, 1], average="macro", zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_true, predictions),
        "Fail Precision": precision_score(y_true, predictions, zero_division=0),
        "Fail Recall": recall_score(y_true, predictions, zero_division=0),
        "Fail F1": f1_score(y_true, predictions, zero_division=0),
        "AP": average_precision_score(y_true, probabilities),
    }


In [ ]:
# All-pass: 모든 샘플을 정상으로 판단하는 교육용 기준입니다.
all_pass_val = np.zeros(len(y_val))
display(pd.DataFrame([
    {"Model": "All-pass", **evaluate_scores(y_val, all_pass_val)}
]).set_index("Model").round(4))
print("불량 Recall과 불량 F1은 0입니다. Macro-F1도 0인지 표에서 확인하세요.")


### Train에서만 학습하는 공통 전처리

아래 helper는 **Train에서 전부 결측이거나 상수인 열을 제거**한 뒤, 선택한 방식으로 결측치를 처리합니다. Validation/Test에는 Train에서 정한 열과 변환을 그대로 적용합니다.

| mode | 처리 | 사용할 모델 |
|---|---|---|
| `native` | NaN을 그대로 유지 | XGBoost |
| `mean` | Train의 센서별 평균으로 대체 | XGBoost / MLP |
| `median` | Train의 센서별 중앙값으로 대체 | XGBoost / MLP |

XGBoost는 이 실습에서 표준화를 요구하지 않습니다. MLP에는 결측치 대체 후 **Train으로 fit한 StandardScaler**를 추가로 적용합니다. 전처리 방법은 자유롭게 확장할 수 있지만, fit에 Validation/Test를 포함하지 않습니다.


In [ ]:
def fit_preprocessor(X_fit, mode):
    if mode not in {"native", "mean", "median"}:
        raise ValueError("mode must be native, mean, or median")
    columns = X_fit.columns[X_fit.nunique(dropna=True) > 1].tolist()
    assert columns, "No usable training features."
    imputer = None
    if mode != "native":
        imputer = SimpleImputer(strategy=mode)
        imputer.fit(X_fit[columns])
    return {"mode": mode, "columns": columns, "imputer": imputer}

def transform_features(X_input, preprocessing):
    values = X_input[preprocessing["columns"]]
    if preprocessing["imputer"] is not None:
        values = preprocessing["imputer"].transform(values)
    return np.asarray(values, dtype=np.float32)


## 6. 실습 A — XGBoost baseline

**질문:** 여러 결정 트리가 앞선 예측의 오류를 보완하도록 차례로 학습되면, 공정 센서와 불량 사이의 관계를 어떻게 표현할 수 있을까요?

XGBoost는 결정 트리를 순차적으로 더해 예측 점수를 만듭니다.

$$F_M(x)=\sum_{m=1}^{M} f_m(x),\qquad p(y=1\mid x)=\sigma(F_M(x)).$$

제공 baseline의 학습 목적함수는 `binary:logistic`입니다. **학습 Loss와 모델 선택 지표는 구분**합니다. 이 실습에서 모델 선택 기준은 항상 **Validation Macro-F1**입니다.

| 설정 | 역할 |
|---|---|
| `n_estimators`, `learning_rate` | 트리 개수와 업데이트 크기 |
| `max_depth`, `min_child_weight` | 트리의 복잡도 |
| `subsample`, `colsample_bytree` | 학습 행·특징의 샘플링 |
| `reg_alpha`, `reg_lambda` | 정규화 |
| `scale_pos_weight` | 학습에서 불량 샘플에 부여하는 가중치 |
| 결정 임계값 | 학습 후 불량 여부를 판단하는 기준 |

먼저 원본과 같은 **150개 트리, 깊이 3, learning rate 0.05, native 결측 처리, 임계값 0.5**를 사용합니다. 가중치 변경은 학습 Loss를, 임계값 변경은 최종 판단 규칙을 바꿉니다.


In [ ]:
BASE_PARAMS = dict(
    objective="binary:logistic", booster="gbtree", tree_method="hist",
    eval_metric="logloss", n_estimators=150, learning_rate=0.05,
    max_depth=3, min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.0, reg_lambda=1.0, scale_pos_weight=1.0,
    random_state=TRAIN_SEED, n_jobs=CONFIG["cpu_threads"], device="cpu",
)

def train_candidate(mode, overrides):
    started = time.perf_counter()
    preprocessing = fit_preprocessor(X_train, mode)
    params = {**BASE_PARAMS, **overrides}
    model = XGBClassifier(**params)
    model.fit(transform_features(X_train, preprocessing), y_train)
    val_prob = model.predict_proba(transform_features(X_val, preprocessing))[:, 1]
    return {"model": model, "preprocessing": preprocessing, "params": params,
            "mode": mode, "val_prob": val_prob, "family": "XGBoost",
            "fit_validation_seconds": time.perf_counter() - started}


In [ ]:
baseline = train_candidate("native", {"scale_pos_weight": 1.0})
display(pd.DataFrame([
    {"Model": "XGBoost baseline", **evaluate_scores(y_val, baseline["val_prob"])}
]).set_index("Model").round(4))
print("XGBoost baseline 준비 완료: Test는 아직 사용하지 않았습니다.")


## 7. 실습 B — PyTorch MLP baseline

**질문:** 트리와 달리, MLP에 결측치 처리와 표준화를 적용하는 이유는 무엇일까요?

MLP는 센서 벡터를 완전연결층과 비선형 함수에 통과시켜 불량 logit 하나를 출력합니다. Sigmoid를 적용하면 임계값과 비교할 예측 점수를 얻습니다.

| 설정 | 제공 baseline |
|---|---|
| 전처리 | Train의 중앙값으로 대체 → Train으로 fit한 표준화 |
| 은닉층 | 128 → 64, 각 층에 ReLU와 dropout 0.3 |
| Loss | `BCEWithLogitsLoss`와 같은 BCE 계산, `pos_weight=1.0` |
| Optimizer | AdamW, learning rate `1e-3`, weight decay `1e-4` |
| 학습 | batch size 64, 최대 60 epoch |
| checkpoint | **Validation Macro-F1 @0.5가 가장 높은 epoch** |
| 장치 | `CONFIG["mlp_device"]`, 기본 CPU |

아래의 구조·Loss·학습 방식은 수정할 수 있습니다. Focal loss 분기도 예제로 제공하지만 사용은 선택입니다. **업데이트는 Train, checkpoint 선택은 Validation**으로 수행합니다.


In [ ]:
MLP_BASE_PARAMS = dict(hidden_sizes=(128, 64), dropout=0.3, epochs=60,
    batch_size=64, learning_rate=1e-3, weight_decay=1e-4,
    pos_weight=1.0, seed=TRAIN_SEED, loss="bce", focal_gamma=2.0)

def make_mlp(input_dim, params):
    layers = []
    for width in params["hidden_sizes"]:
        layers += [nn.Linear(input_dim, width), nn.ReLU(), nn.Dropout(params["dropout"])]
        input_dim = width
    return nn.Sequential(*layers, nn.Linear(input_dim, 1))

def mlp_loss(logits, targets, params):
    # 적절한 이진 분류 Loss로 자유롭게 수정할 수 있습니다.
    if params["loss"] == "bce":
        return nn.functional.binary_cross_entropy_with_logits(logits, targets,
            pos_weight=logits.new_tensor(params["pos_weight"]))
    if params["loss"] == "focal":
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        prob = torch.sigmoid(logits)
        pt = torch.where(targets == 1, prob, 1 - prob)
        weights = torch.where(targets == 1, params["pos_weight"], 1.0)
        return (weights * (1 - pt).pow(params["focal_gamma"]) * bce).mean()
    raise ValueError("Implement your custom loss here or use bce/focal.")

def train_mlp(mode="median", overrides=None):
    started = time.perf_counter()
    params = {**MLP_BASE_PARAMS, **(overrides or {})}
    torch.manual_seed(params["seed"])
    # 기본 CPU 경로는 원본과 동일하며, CUDA는 선택입니다.
    torch.set_num_threads(CONFIG["cpu_threads"])
    device = MLP_DEVICE
    if device.type == "cuda":
        torch.cuda.manual_seed_all(params["seed"])
    preprocessing = fit_preprocessor(X_train, mode)
    assert mode in {"mean", "median"}, "MLP requires finite inputs; impute first."
    train_array = transform_features(X_train, preprocessing)
    val_array = transform_features(X_val, preprocessing)
    scaler = StandardScaler().fit(train_array)
    preprocessing["scaler"] = scaler
    tx = torch.tensor(scaler.transform(train_array), dtype=torch.float32)
    vx = torch.tensor(scaler.transform(val_array), dtype=torch.float32)
    ty = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
    assert torch.isfinite(tx).all() and torch.isfinite(vx).all()
    loader = DataLoader(TensorDataset(tx, ty), batch_size=params["batch_size"], shuffle=True,
        generator=torch.Generator().manual_seed(params["seed"]))
    vx = vx.to(device)
    model = make_mlp(tx.shape[1], params).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["learning_rate"],
                                 weight_decay=params["weight_decay"])
    history, best_score, best_state, best_epoch = [], -1.0, None, None
    for epoch in range(params["epochs"]):
        model.train()
        total_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = mlp_loss(model(xb).squeeze(1), yb, params)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(yb)
        model.eval()
        with torch.no_grad():
            val_prob = torch.sigmoid(model(vx).squeeze(1)).cpu().numpy()
        # 기본 checkpoint 선택: 임계값 0.5에서 Validation Macro-F1 최대화.
        # 선택 방식을 바꿀 수 있지만 Test로 선택하지 않습니다.
        score = evaluate_scores(y_val, val_prob)[MAIN_METRIC]
        history.append({"Epoch": epoch + 1, "Train loss": total_loss / len(ty),
                        "Validation Macro-F1 @0.5": score})
        if score > best_score:
            best_score, best_epoch = score, epoch + 1
            best_state = copy.deepcopy(model.state_dict())
    assert best_state is not None
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_prob = torch.sigmoid(model(vx).squeeze(1)).cpu().numpy()
    return {"model": model, "preprocessing": preprocessing, "params": params,
            "mode": mode, "family": "MLP", "val_prob": val_prob,
            "history": history, "best_epoch": best_epoch,
            "fit_validation_seconds": time.perf_counter() - started,
            "training_device": str(device)}


In [ ]:
mlp_baseline = train_mlp("median", {"pos_weight": 1.0})
display(pd.DataFrame([
    {"Model": "MLP baseline", **evaluate_scores(y_val, mlp_baseline["val_prob"])}
]).set_index("Model").round(4))
print("Validation으로 선택한 epoch:", mlp_baseline["best_epoch"])


## 8. 첫 번째 비교 — 같은 Validation에서 두 모델 평가

**질문:** Accuracy가 가장 높은 모델과 Macro-F1이 가장 높은 모델이 항상 같을까요?

All-pass, XGBoost, MLP를 **동일한 Validation, 동일한 임계값 0.5**에서 비교합니다. MLP의 학습 Loss가 계속 내려가더라도 Validation Macro-F1은 달라질 수 있습니다. 학습곡선과 불량 Recall을 함께 확인하세요.

이 비교는 자유 실험의 출발점입니다. 아직 최종 모델이나 Test 점수를 확정하는 단계는 아닙니다.


In [ ]:
validation_baselines = pd.DataFrame([
    {"Model": "All-pass", **evaluate_scores(y_val, all_pass_val)},
    {"Model": "XGBoost baseline", **evaluate_scores(y_val, baseline["val_prob"])},
    {"Model": "MLP baseline", **evaluate_scores(y_val, mlp_baseline["val_prob"])},
]).set_index("Model")
display(validation_baselines.round(4))
history = pd.DataFrame(mlp_baseline["history"])
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
history.plot(x="Epoch", y="Train loss", ax=axes[0])
history.plot(x="Epoch", y="Validation Macro-F1 @0.5", ax=axes[1])
plt.tight_layout()
save_output("SECOM_mlp_baseline_history.png", lambda path: fig.savefig(path, dpi=160, bbox_inches="tight"))
plt.show()
save_output("SECOM_validation_baselines.csv", lambda path: validation_baselines.to_csv(path))
save_output("SECOM_mlp_baseline_history.csv", lambda path: history.to_csv(path, index=False))
print("MLP baseline 선택 epoch:", mlp_baseline["best_epoch"])
print("All-pass의 Accuracy와 불량 Recall을 함께 읽어 보세요.")


## 9. 실습 C — Validation Macro-F1 자유 개선

**XGBoost와 MLP를 모두 비교한 뒤, 둘 중 원하는 모델로 Validation Macro-F1을 최대화하세요.** 특정 모델 사용에 따른 우대는 없습니다.

아래 항목은 **실험 아이디어**입니다. 필수 비교나 최소 실험 횟수가 아니며, 다른 방법을 선택해도 됩니다. Loss 변경을 포함해 모델·학습 방식을 자유롭게 수정할 수 있습니다.

| 변경할 수 있는 항목 | 예시 | 생각해 볼 질문 |
|---|---|---|
| 결측치 처리·특징 | native / mean / median, 특징 선택 | 어떤 센서의 정보가 유용한가? |
| 불균형 대응 | class weight, Train 내부 샘플링 | 불량 Recall과 정상 오탐이 어떻게 변하는가? |
| XGBoost | 깊이, 트리 수, learning rate, 정규화, objective | 복잡도와 과적합의 관계는? |
| MLP | 은닉층, dropout, BCE / focal / 사용자 Loss | 구조와 Loss가 불량 탐지에 어떤 영향을 주는가? |
| 학습 방식 | optimizer, scheduler, epoch, checkpoint, 학습 seed | Validation 성능이 안정적인가? |
| 결정 임계값 | 0.5 유지 또는 Validation에서 탐색 | 불량 누락과 정상 오탐의 균형은? |

**변경하지 않는 것:** 데이터 분할, 라벨 매핑, 평가 정의, Train-only 학습, Test 최종 평가 원칙. Validation을 합쳐 재학습하지 않습니다.

아래는 수정·교체 가능한 편의 코드입니다. 추가 실험 목록은 기본적으로 비어 있고, 임계값 탐색도 꺼져 있습니다. 제공 baseline의 실행만으로 과제가 완료되지는 않으므로 **자신이 선택한 변경과 이유, 결과를 보고서에 설명**하세요.


In [ ]:
# 자유롭게 수정할 실험 설정. 주석 예시는 필수 실험이 아닙니다.
positive_weight = float((y_train == 0).sum() / (y_train == 1).sum())
print("Train N_pass / N_fail:", positive_weight)

TUNE_THRESHOLD = False  # 선택: True이면 Validation에서만 임계값 탐색
THRESHOLDS = np.linspace(0.01, 0.99, 99) if TUNE_THRESHOLD else np.array([0.5])

EXPERIMENTS = []
# 추가 예시 (원할 때 주석 해제 / 수정):
# EXPERIMENTS = [
#     {"name": "XGB weighted", "mode": "native",
#      "params": {"scale_pos_weight": positive_weight, "max_depth": 3}},
# ]

MLP_EXPERIMENTS = []
# MLP_EXPERIMENTS = [
#     {"name": "MLP weighted", "mode": "median",
#      "params": {"pos_weight": positive_weight, "hidden_sizes": (128, 64)}},
# ]

assert np.all((THRESHOLDS > 0) & (THRESHOLDS < 1))
all_names = ["XGBoost baseline", "MLP baseline"] + [
    spec["name"] for spec in EXPERIMENTS + MLP_EXPERIMENTS]
assert len(set(all_names)) == len(all_names), "실험 이름은 서로 달라야 합니다."


In [ ]:
def select_threshold(y_true, probabilities):
    rows = [{"Threshold": float(t), **evaluate_scores(y_true, probabilities, t)}
            for t in THRESHOLDS]
    table = pd.DataFrame(rows)
    ranked = table.assign(distance_to_half=(table["Threshold"] - 0.5).abs()).sort_values(
        [MAIN_METRIC, "distance_to_half", "Threshold"], ascending=[False, True, True])
    return float(ranked.iloc[0]["Threshold"]), table


In [ ]:
# 두 baseline 모두 최종 선택 후보입니다.
candidates = [{**baseline, "name": "XGBoost baseline"},
              {**mlp_baseline, "name": "MLP baseline"}]
for spec in EXPERIMENTS:
    candidate = train_candidate(spec["mode"], spec["params"])
    candidate["name"] = spec["name"]
    candidates.append(candidate)
    print("Trained:", spec["name"])

for spec in MLP_EXPERIMENTS:
    candidate = train_mlp(spec["mode"], spec["params"])
    candidate["name"] = spec["name"]
    candidates.append(candidate)
    print("Trained:", spec["name"], "| Selected epoch:", candidate["best_epoch"])
assert len({c["name"] for c in candidates}) == len(candidates)

records = []
for candidate in candidates:
    threshold, threshold_table = select_threshold(y_val, candidate["val_prob"])
    candidate["threshold"] = threshold
    candidate["threshold_table"] = threshold_table
    candidate["selection_score"] = evaluate_scores(
        y_val, candidate["val_prob"], threshold)[MAIN_METRIC]
    for label, t in [("Default 0.5", 0.5), ("Validation-selected", threshold)]:
        records.append({"Experiment": candidate["name"], "Family": candidate["family"], "Missing": candidate["mode"],
            "Decision": label, "Threshold": t,
            "Parameters": str(candidate["params"]),
            "Fit+validation seconds": candidate.get("fit_validation_seconds", np.nan),
            **evaluate_scores(y_val, candidate["val_prob"], t)})
experiment_results = pd.DataFrame(records)
display(experiment_results.sort_values(MAIN_METRIC, ascending=False, kind="stable").round(4))
save_output("SECOM_validation_experiments.csv", lambda path: experiment_results.to_csv(path, index=False))
best_candidate = max(candidates, key=lambda c: c["selection_score"])
print("Selected:", best_candidate["name"], "| Threshold:", best_candidate["threshold"])
print("Validation Macro-F1:", best_candidate["selection_score"])
if not EXPERIMENTS and not MLP_EXPERIMENTS and not TUNE_THRESHOLD:
    print("현재 설정은 두 baseline 비교입니다. 직접 선택한 추가 실험·해석을 이어서 작성하세요.")
print("시간 열은 전처리·학습·Validation 평가를 포함한 벽시계 시간입니다.")
print("두 학습기의 검증 빈도와 장치가 다를 수 있어 순수 학습 속도의 동일 조건 비교가 아닙니다.")


### 계열별 최선의 모델과 전체 최종 후보 확인

각 계열에서 Validation Macro-F1이 가장 높은 설정을 고르고, 전체 후보 중 최종 모델 하나를 선택합니다. 아래 혼동행렬의 **불량 누락(FN)**과 **정상 오탐(FP)**을 확인하세요.

임계값 탐색을 껐다면 `Default 0.5`와 `Validation-selected` 행이 같은 것은 정상입니다. 모델 점수가 같으면 현재 후보 목록에서 먼저 나온 후보를 유지합니다. 선택 규칙을 바꾸려면 Test를 보기 전에 정하세요.


In [ ]:
# 계열별 Validation 최선의 설정을 비교합니다.
best_by_family = {family: max((c for c in candidates if c["family"] == family),
    key=lambda c: c["selection_score"]) for family in ["XGBoost", "MLP"]}
family_comparison = pd.DataFrame([
    {"Family": family, "Experiment": candidate["name"], "Threshold": candidate["threshold"],
     **evaluate_scores(y_val, candidate["val_prob"], candidate["threshold"])}
    for family, candidate in best_by_family.items()
]).set_index("Family")
display(family_comparison.round(4))
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (family, candidate) in zip(axes, best_by_family.items()):
    ConfusionMatrixDisplay.from_predictions(y_val,
        (candidate["val_prob"] >= candidate["threshold"]).astype(int),
        labels=[0, 1], display_labels=["Pass", "Fail"], ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Validation: best {family}")
plt.tight_layout()
save_output("SECOM_validation_confusion_matrices.png", lambda path: fig.savefig(path, dpi=160, bbox_inches="tight"))
plt.show()
print("Final family selected by Validation Macro-F1:", best_candidate["family"])
save_output("SECOM_validation_family_comparison.csv", lambda path: family_comparison.to_csv(path))


## 10. 선택 분석 — 임계값과 혼동행렬

**질문:** 같은 모델의 예측 점수를 사용하면서 임계값만 낮추면, 불량 Recall과 정상 오탐은 어떻게 달라질까요?

이 분석은 **선택 사항**입니다. `TUNE_THRESHOLD=False`이면 현재 임계값 한 점만 표시합니다. `True`이면 0.01~0.99의 후보 중 Validation Macro-F1이 가장 높은 값을 고릅니다. 동률이면 0.5에 가까운 값, 그다음 작은 값을 선택합니다.

표시된 곡선과 혼동행렬은 모두 **Validation 결과**입니다. Test를 보고 임계값을 다시 고르면 안 됩니다.


In [ ]:
table = best_candidate["threshold_table"]
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for metric in ["Accuracy", "Macro-F1", "Fail Recall"]:
    axes[0].plot(table["Threshold"], table[metric], label=metric, marker="o" if len(table) == 1 else None)
axes[0].axvline(best_candidate["threshold"], color="black", linestyle="--", label="Selected")
axes[0].set(xlabel="Threshold", ylabel="Validation score", title="Threshold trade-offs")
axes[0].legend()
for ax, threshold in zip(axes[1:], [0.5, best_candidate["threshold"]]):
    pred = (best_candidate["val_prob"] >= threshold).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_val, pred, labels=[0, 1], display_labels=["Pass", "Fail"],
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Validation: threshold={threshold:.2f}")
plt.tight_layout()
save_output("SECOM_validation_thresholds.png", lambda path: fig.savefig(path, dpi=160, bbox_inches="tight"))
plt.show()


## 11. 최종 설정 확정 — Test를 보기 전에 고정하기

**실습 A~C를 마치고 모델·전처리·Loss·학습 설정·checkpoint·임계값을 모두 선택한 뒤** 아래 두 스위치를 `True`로 바꿉니다.

- `FINAL_SETTINGS_CONFIRMED`: Validation 결과만으로 모든 선택을 끝냈다는 표시.
- `RUN_FINAL_TEST`: 다음 절의 최종 Test 평가를 실행하는 스위치.

전체 최종 모델, 계열별 최선의 모델, 두 원본 baseline을 복사해 고정합니다. **Train + Validation으로 재학습하지 않습니다.** 최종 설정은 JSON으로 기록하고, MLP의 고정 복사본은 CPU로 옮겨 보관합니다.

> Test는 결과보고용입니다. 세션 guard와 완료 파일은 실수로 반복 평가하는 것을 줄이는 장치이며, 비공개 테스트 서버나 보안 장치는 아닙니다. Test 결과를 본 뒤 선택을 바꾸지 마세요.


In [ ]:
def predict_candidate(candidate, X_input):
    values = transform_features(X_input, candidate["preprocessing"])
    if candidate["family"] == "XGBoost":
        return candidate["model"].predict_proba(values)[:, 1]
    values = candidate["preprocessing"]["scaler"].transform(values)
    candidate["model"].eval()
    device = next(candidate["model"].parameters()).device
    with torch.no_grad():
        return torch.sigmoid(candidate["model"](
            torch.tensor(values, dtype=torch.float32, device=device)).squeeze(1)).cpu().numpy()


In [ ]:
FINAL_SETTINGS_CONFIRMED = False
RUN_FINAL_TEST = False

FINAL_DONE_PATH = RUN_DIR / "SECOM_final_completed.json"

def freeze_candidate(candidate):
    frozen = copy.deepcopy(candidate)
    if frozen["family"] == "MLP":
        frozen["model"] = frozen["model"].cpu().eval()
    return frozen

if not RUN_FINAL_TEST:
    print("최종 평가는 대기 중입니다. 자유 실험을 마친 뒤 두 스위치를 True로 바꾸세요.")
elif not FINAL_SETTINGS_CONFIRMED:
    raise RuntimeError("Validation으로 설정을 확정한 뒤 FINAL_SETTINGS_CONFIRMED=True로 바꾸세요.")
elif FINAL_DONE_PATH.exists() or globals().get("_final_test_evaluated", False):
    print("이미 완료한 Test 평가가 있습니다. 다음 절에서 저장된 결과를 표시합니다.")
else:
    final_bundle = freeze_candidate(best_candidate)
    frozen_by_family = {family: freeze_candidate(candidate)
                        for family, candidate in best_by_family.items()}
    frozen_baselines = {"XGBoost": freeze_candidate(baseline),
                        "MLP": freeze_candidate(mlp_baseline)}
    frozen_selection = {
        "experiment": final_bundle["name"], "family": final_bundle["family"],
        "threshold": final_bundle["threshold"], "parameters": final_bundle["params"],
        "best_epoch": final_bundle.get("best_epoch"),
        "validation_macro_f1": final_bundle["selection_score"],
        "data_hashes": DATA_HASHES, "split_seed": SPLIT_SEED, "split_hash": SPLIT_HASH,
        "config": copy.deepcopy(CONFIG), "versions": VERSIONS,
        "selected_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    (RUN_DIR / "SECOM_frozen_selection.json").write_text(
        json.dumps(frozen_selection, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print("확정 모델:", final_bundle["name"], "| 계열:", final_bundle["family"])
    print("확정 임계값:", final_bundle["threshold"])
    print("확정 파라미터:", final_bundle["params"])
    print("다음 절의 Test 셀을 실행하세요. 재학습은 하지 않습니다.")


## 12. 실습 D — 최종 Test 1회 평가와 결과 저장

최종 표에는 원본 과제와 같은 **7개 비교 행**이 표시됩니다.

1. All-pass
2. XGBoost baseline @0.5
3. MLP baseline @0.5
4. Validation으로 고른 최선의 XGBoost + 해당 임계값
5. Validation으로 고른 최선의 MLP + 해당 임계값
6. 전체 최종 모델 @0.5
7. **전체 최종 모델 + Validation에서 확정한 임계값**

**최종 점수는 7번 행의 Test Macro-F1**입니다. Test 표에서 가장 높은 행을 보고 최종 모델을 다시 선택하지 않습니다. 6번과 7번은 동일한 모델에서 임계값의 영향을 구분하기 위한 비교입니다.

평가가 완료되면 `SECOM_final_completed.json`을 남깁니다. 같은 폴더에서 다시 실행하면 **기존 Test CSV·혼동행렬·분류 보고서를 표시**하며 재평가하지 않습니다. 완료된 폴더의 설정·Validation 표·그림도 덮어쓰지 않아 기존 최종 모델과 실행 기록을 함께 보존합니다. 이미 본 Test 점수를 이용해 다시 튜닝하지 마세요.


In [ ]:
# 고정 Test 평가. 지표·라벨·비교 행을 유지하세요.
if not RUN_FINAL_TEST:
    print("Test는 아직 실행하지 않았습니다. 모든 선택을 마친 뒤 11절의 두 스위치를 True로 바꾸세요.")
elif not FINAL_SETTINGS_CONFIRMED:
    raise RuntimeError("최종 설정 확인이 필요합니다. 11절을 먼저 실행하세요.")
elif FINAL_DONE_PATH.exists():
    completion = json.loads(FINAL_DONE_PATH.read_text(encoding="utf-8"))
    if completion.get("data_hashes") != DATA_HASHES or completion.get("split_hash") != SPLIT_HASH:
        raise RuntimeError("저장된 Test 결과의 데이터/분할이 현재 실행과 다릅니다. 기존 기록을 확인하세요.")
    print("저장된 최종 결과를 표시합니다. 현재 후보로 Test를 다시 평가하지 않습니다.")
    print("저장된 모델:", completion["experiment"], "| 임계값:", completion["threshold"])
    test_results = pd.read_csv(RUN_DIR / "SECOM_final_test_results.csv", index_col=0)
    display(test_results.round(4))
    report_path = RUN_DIR / "SECOM_final_classification_report.txt"
    if report_path.exists():
        print(report_path.read_text(encoding="utf-8"))
    matrix_path = RUN_DIR / "SECOM_final_confusion_matrices.png"
    if matrix_path.exists():
        display(Image(filename=str(matrix_path)))
elif globals().get("_final_test_evaluated", False):
    print("이 세션에서 이미 평가했습니다. 위의 결과를 사용하세요.")
else:
    if "final_bundle" not in globals():
        raise RuntimeError("11절의 설정 확정 셀부터 실행하세요.")
    X_test, y_test = X.iloc[test_idx].copy(), y.iloc[test_idx].copy()
    final_prob = predict_candidate(final_bundle, X_test)
    final_cases = [("All-pass", np.zeros(len(y_test)), 0.5)]
    for family, candidate in frozen_baselines.items():
        final_cases.append((f"{family} baseline @0.5", predict_candidate(candidate, X_test), 0.5))
    for family, candidate in frozen_by_family.items():
        final_cases.append((f"Best {family} (validation-selected)",
            predict_candidate(candidate, X_test), candidate["threshold"]))
    final_cases += [
        ("Selected final model @0.5", final_prob, 0.5),
        (f"FINAL: {final_bundle['family']} + validation threshold", final_prob, final_bundle["threshold"]),
    ]
    test_results = pd.DataFrame([
        {"Model": name, "Threshold": t, **evaluate_scores(y_test, prob, t)}
        for name, prob, t in final_cases
    ]).set_index("Model")
    _final_test_evaluated = True
    display(y_test.value_counts().sort_index().rename("Test class count").to_frame())
    display(test_results.round(4))
    final_predictions = (final_prob >= final_bundle["threshold"]).astype(int)
    final_report = classification_report(y_test, final_predictions, labels=[0, 1],
                                         target_names=["Pass", "Fail"], zero_division=0)
    print(final_report)
    (RUN_DIR / "SECOM_final_classification_report.txt").write_text(final_report, encoding="utf-8")
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, case in zip(axes, [final_cases[3], final_cases[4], final_cases[-1]]):
        name, prob, t = case
        ConfusionMatrixDisplay.from_predictions(y_test, (prob >= t).astype(int),
            labels=[0, 1], display_labels=["Pass", "Fail"], ax=ax, colorbar=False, cmap="Blues")
        ax.set_title(name.replace(" (validation-selected)", "").replace(" + validation threshold", ""))
    plt.tight_layout()
    fig.savefig(RUN_DIR / "SECOM_final_confusion_matrices.png", dpi=160, bbox_inches="tight")
    plt.show()
    checkpoint = {
        "model": final_bundle["model"], "family": final_bundle["family"],
        "best_epoch": final_bundle.get("best_epoch"), "preprocessing": final_bundle["preprocessing"],
        "threshold": final_bundle["threshold"], "parameters": final_bundle["params"],
        "experiment": final_bundle["name"], "validation_macro_f1": final_bundle["selection_score"],
        "split_seed": SPLIT_SEED, "train_idx": train_idx, "val_idx": val_idx, "test_idx": test_idx,
        "label_mapping": {-1: 0, 1: 1}, "test_results": test_results,
        "versions": VERSIONS, "environment": ENVIRONMENT,
        "data_hashes": DATA_HASHES, "split_hash": SPLIT_HASH,
    }
    joblib.dump(checkpoint, RUN_DIR / "SECOM_final_model.joblib")
    test_results.to_csv(RUN_DIR / "SECOM_final_test_results.csv")
    print("Final Test Macro-F1:", test_results.iloc[-1][MAIN_METRIC])
    print("Saved: SECOM_final_model.joblib / SECOM_final_test_results.csv")
    FINAL_DONE_PATH.write_text(json.dumps({
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment": final_bundle["name"], "family": final_bundle["family"],
        "threshold": final_bundle["threshold"],
        "validation_macro_f1": final_bundle["selection_score"],
        "test_macro_f1": float(test_results.iloc[-1][MAIN_METRIC]),
        "data_hashes": DATA_HASHES, "split_hash": SPLIT_HASH,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print("최종 평가 완료. 이후 Test 결과는 보고서 작성에만 사용하세요.")


### 저장된 결과 확인

| 파일 | 용도 |
|---|---|
| `SECOM_validation_baselines.csv` | 두 baseline과 All-pass의 Validation 비교 |
| `SECOM_validation_experiments.csv` | 자유 실험 설정·임계값·Validation 지표 |
| `SECOM_validation_family_comparison.csv` | 계열별 최선의 Validation 결과 |
| `SECOM_final_test_results.csv` | 확정된 모델들의 최종 Test 비교 |
| `SECOM_final_model.joblib` | 최종 모델, 전처리, 임계값, 분할 인덱스, 버전 및 결과 |
| `SECOM_frozen_selection.json` | Test 전에 선택한 최종 설정 |
| `SECOM_final_completed.json` | 최종 평가 완료 기록 |
| `SECOM_split_indices.npz`, `SECOM_data_hashes.json` | 같은 데이터·분할인지 확인하기 위한 기록 |
| `SECOM_environment.json`, `*.png`, 학습 기록 CSV | 실행 환경과 결과 해석 자료 |

노트북의 출력은 지우지 않고 저장합니다. 최종 모델과 실험 CSV는 검증을 위해 함께 보관하세요. 이 노트북에는 사전에 계산한 결과나 예시 성능을 채워 넣지 않았습니다.


In [ ]:
saved_files = [{"file": path.name, "size_kb": round(path.stat().st_size / 1024, 2)}
               for path in sorted(RUN_DIR.glob("SECOM_*")) if path.is_file()]
if saved_files:
    display(pd.DataFrame(saved_files))
else:
    print("앞의 실습 셀을 실행하면 결과 파일이 생성됩니다.")


## 13. 결과보고서 — 수치와 이유를 함께 설명하기

**자신이 수행한 실험을 바탕으로 이론적·실증적 분석, 정량·정성 결과를 작성하세요.** 아래는 작성 가이드이며, 새로운 필수 실험 목록이나 최소 실험 횟수가 아닙니다. 필요에 맞게 구성해도 됩니다.

### 13.1 문제 이해와 두 모델의 차이

- 왜 Accuracy만으로 불량 분류 성능을 판단하기 어려운가요?
- XGBoost와 MLP의 학습 방식, 결측치 처리, 표준화의 차이를 설명하세요.

**작성:**

> 여기에 작성하세요.

### 13.2 자유 실험의 가설·변경·결과

어떤 개선을 기대했고, 실제로 무엇을 바꿨으며, Validation 결과가 어떻게 변했는지 설명하세요. 수행한 실험만 작성합니다.

| 수행한 실험 | 변경 이유 / 가설 | 바꾼 설정 | Validation Macro-F1 | 관찰한 결과 |
|---|---|---|---|---|
| 작성 | 작성 | 작성 | 실제 결과 | 작성 |

**해석:**

> 여기에 작성하세요. 학습 Loss와 평가 지표의 차이, 과적합, 임계값, FN/FP 변화 등 자신의 실험과 관련된 근거를 연결합니다.

### 13.3 최종 선택과 Test 결과

- 최종 모델 계열, 실험 이름, 전처리, 핵심 학습 설정, 임계값을 기록하세요.
- 왜 이 모델을 선택했는지 **Validation 결과**로 설명하세요.
- 확정된 모델의 Test Macro-F1과 보조 지표를 보고하고, 두 모델의 결과 및 혼동행렬을 비교하세요.

**작성:**

> 여기에 작성하세요. Test 표에서 사후에 다른 모델을 고르지 않습니다.

### 13.4 제조 현장 관점의 해석과 한계

불량 누락(FN)과 정상 오탐(FP)이 어떤 비용을 만들 수 있는지, 이번 실험이 확인한 것과 확인하지 못한 것을 설명하세요. 무작위 분할 결과를 미래 공정에 대한 검증으로 해석하지 않습니다. 추가 개선 아이디어는 제안할 수 있으나, **최종 Test 결과를 이용한 후속 튜닝은 수행하지 않습니다**.

**작성:**

> 여기에 작성하세요.


### 참고자료

- [UCI SECOM](https://archive.ics.uci.edu/dataset/179/secom) — 데이터 및 라벨 정보
- [XGBoost parameters](https://xgboost.readthedocs.io/en/stable/parameter.html) — 목적함수, 트리 설정, class weight
- [scikit-learn F1 score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html) — 클래스별 F1과 macro 평균
- [scikit-learn Average Precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html) — 연속 점수 기반 AP
- [scikit-learn: Data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage) — Train-only 전처리
- [PyTorch 설치 안내](https://pytorch.org/get-started/locally/) · [CUDA semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html) — 로컬 실행 환경과 장치 사용

**구성 참고:** `Nsquare_MultiHopRAG_5Methods_RTX5080.ipynb`의 한국어 설명, 공통 설정표, 실습별 질문·코드·결과 해석, 명시적 최종 평가 흐름을 적용했습니다.
